In [ ]:
import numpy as np
from sklearn.metrics import pairwise_distances

def calculate_pertinences(
    x : np.ndarray,
    c : np.ndarray,
):
    dist = pairwise_distances(x, c, metric="euclidean")
    pert = 1/dist
    pert /= pert.sum(axis=1, keepdims=True)
    return pert
    

In [ ]:
from sklearn.datasets import make_blobs

centers = np.array(
    [
        [1, 1],
        #[1, -1],
        [-1, -1],
        #[-1, 1],
        #[0,0],
    ]
)

X, y = make_blobs(
    n_samples=1000,
    centers=centers,
    n_features=2,
    cluster_std=0.2,
    random_state=42,
)

print(X.shape, y.shape)

In [ ]:
y

In [ ]:
pert = calculate_pertinences(X, centers)
pert

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, p_train, p_test = train_test_split(
    X,
    y,
    pert,
    test_size=0.2,
    random_state=42,
)

In [ ]:
import torch 

x_tensor = torch.tensor(X_train).float()
y_tensor = torch.tensor(y_train)
p_tensor = torch.tensor(p_train).float()

print(x_tensor.shape, y_tensor.shape, p_tensor.shape)

In [ ]:
from mlxp import models
from mlxp import training

clf = models.MLP(
    input_dim = 2,
    hidden_dim = 100,
    output_dim = 2,
    n_layers = 20,
    dropout = 0.0,
    block_type = "simple" # "simple" or "residual"
)

print(clf)

In [ ]:
import torch.nn as nn

train_data = training.train_mlp(
    model=clf,
    x=x_tensor,
    y=y_tensor,
    p=y_tensor,
    optimizer_type="adam",
    val_split=0.2,
    epochs=100,
    seed=42,
    loss_fn=nn.CrossEntropyLoss(),
    patience=1000000,
    verbose=True,
    lr=1e-3,
    task="classification",
    record_nc=True,
)

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np 

plt.figure(figsize=(7,3))
plt.plot(train_data['train_loss'], 'b', label='Train loss')
plt.plot(train_data['val_loss'], 'r', label='Val loss')
plt.plot(1-np.array(train_data['train_acc']), 'b:', label='Train error rate')
plt.plot(1-np.array(train_data['val_acc']), 'r:', label='Val error rate')
plt.plot(train_data['train_nc1'], 'b--', label='Train NC2')
plt.plot(train_data['val_nc1'], 'r--', label='Val NC2')
plt.ylabel('Cross Entropy or \n 1-accuracy')
plt.xlabel('Epochs')
plt.legend()
#plt.semilogy()
plt.show()

In [ ]:
plt.figure(figsize=(7,3))

equinorm = [a['equinorm'] for a in train_data['train_nc2']]
cos_std = [a['cos_std'] for a in train_data['train_nc2']]
cos_mean = [a['cos_mean'] for a in train_data['train_nc2']]
plt.plot(equinorm, label='equinorm')
plt.plot(cos_std, label='cos_std')
plt.plot(cos_mean, label='cos_mean')
plt.legend()

plt.show()

In [ ]:
from mlxp.models import get_embeddings

z = get_embeddings(model=clf, x=x_tensor)

In [ ]:
from sklearn.decomposition import PCA

z_pca = PCA(n_components=2).fit_transform(
    z.detach().cpu().numpy()
)

#z_pca

In [ ]:


plt.figure(figsize=(6, 5))


plt.scatter(
    z_pca[:, 0],
    z_pca[:, 1],
    c=y_tensor.detach().cpu().numpy(),
    cmap="viridis",
    alpha=0.8,
)

plt.gca().collections[0].set_cmap(plt.colormaps["tab10"].resampled(5))
plt.xlabel("Embedding 1")
plt.ylabel("Embedding 2")
plt.title("Scatterplot of z colored by labels")
#plt.legend(label="Label")
plt.show()